# ESM2 as-standard model

Notebook draft to generate ESM2 embeddings from pylogeny-aware data. 

Tasks
- Import ESM checkpoint
- Import ESM tokenizer
- Import input data (target)
- Preprocess data
- Tokenize data
- Generate embeddings

Downstream tasks
- Run embeddings through classification head pre-trained using lower-level data for token classification
- Assess performance


In [1]:
# Dependancies and libraries
import torch
import esm
from transformers import AutoModel, AutoTokenizer
import pandas as pd

In [2]:
# ESM checkpoints
ESM = ['facebook/esm2_t48_15B_UR50D',
        'facebook/esm2_t36_3B_UR50D',
        'facebook/esm2_t33_650M_UR50D',
        'facebook/esm2_t30_150M_UR50D',
        'facebook/esm2_t12_35M_UR50D',
        'facebook/esm2_t6_8M_UR50D']

In [3]:
# Define checkpoint to be used
checkpoint = ESM[5]

In [4]:
# Create tokenizer and model objects
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModel.from_pretrained(checkpoint)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [5]:
# Import target data
df_target= pd.read_csv("/Users/harry/Documents/Data Science MSc/PROJECT/MScProject/Target_1769.csv")

# Remove extraneous columns
df_target = df_target.iloc[:,0:5]

df_target


,Info_protein_id,Info_pos,Info_AA,Info_group,Class
0,P24301.2,1,M,621.0,1.0
1,P24301.2,2,A,621.0,1.0
2,P24301.2,3,K,621.0,1.0
3,P24301.2,4,V,621.0,1.0
4,P24301.2,5,K,621.0,1.0
...,...,...,...,...,...
9016,O33084.3,96,S,622.0,-1.0
9017,O33084.3,97,K,622.0,-1.0
9018,O33084.3,98,M,622.0,-1.0
9019,O33084.3,99,N,622.0,-1.0


In [6]:
# Aggreate rows and update format
df_target = df_target.groupby(['Info_protein_id', 'Info_group']).agg(
    sequence=('Info_AA', ''.join), 
    label=('Class', list), 
    position=('Info_pos', list))

In [7]:
df_target

,,sequence,label,position
Info_protein_id,Info_group,,,
AAA17319.1,606.0,MSWKSVGRCDAEKRLQYARKHYQIPLIREPRNRVKQTAASHQSPCA...,"[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
AAA17366.1,442.0,MALWRSLMKRPNLIIDVGMHNGQDTAFYLAKGFDVVALEANPVLVD...,"[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
AAA25354.1,145.0,MPGRDGETQPASCGRPSRALHPASVSNGGCRHPVTLASFLIRRNHF...,"[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
AAA25362.1,132.0,MARAVGIDLGTTNSVVSVLEGGDPVVVANSEGSRTTPSTVAFARNG...,"[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
AWV47686.1,608.0,MARTLRCCAPHNIAPSNRRPAGRCHSLISLLHREIYQVQQEKNRPD...,"[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
AWV48171.1,646.0,MLGPSIGAYPDRHDSSDKIEASLRHLPRSSEAAGDRAGRGQDRSSS...,"[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
CAA37572.1,634.0,EFGLDGVTYEIDLTNKNAAKLRGDLRQWVSAGRRVGGRRRGRSNSG...,"[nan, nan, nan, nan, nan, 1.0, 1.0, 1.0, 1.0, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
CAA43269.1,405.0,MIDVSGKIRAWGRWLLVGAAATLPSLISLAGGAATASAFSRPGLPV...,"[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
CAA46515.1,498.0,MTDQPPPSGSNPTPAPPPPGSSGGYEPSFAPSELGSAYPPPTAPPV...,"[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."


In [8]:
# Create a list of sequences
sequences = df_target['sequence'].tolist()

# Instantiate the tokenizer using the AA sequence lists as input to the tokenizer to create tokenized sequences
inputs = tokenizer(
    sequences,
    padding=True,
    truncation=True,
    max_length=1024,
    return_tensors="pt"
)

# Put the model into evaluation mode
model.eval()

# Generate embeddings for the 
with torch.inference_mode():
    outputs = model(**inputs)

In [9]:
outputs[0].size()

torch.Size([21, 1024, 320])